In [2]:
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from core import *
from utils import *
from lark import Tree, Token
from pm4py import save_vis_petri_net
import pandas as pd
import sys
#print(f"Versione Python: {sys.version}")

#print(torch.cuda.is_available())
#print(torch.__version__)

# SETTINGS
NARY = 1
CURRENT_STRING = SEED_STRING
PROBABILITIES = 0.34, 0.33, 0.33
FILE_PATH_PNG = "petri_net_output.png"
TRACE_ENC_REG = "data/regions_full.csv"
TRACE_ENC_TAS = "data/tasks_full.csv"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
learning_rate = 3e-4

In [3]:
'''def get_batch(split, train_data_X, train_data_Y, val_data_X, val_data_Y, batch_size, device):
    # Seleziona i dati corretti
    X_data = train_data_X if split == 'train' else val_data_X
    Y_data = train_data_Y if split == 'train' else val_data_Y

    # Genera indici casuali
    ix = torch.randint(len(X_data), (batch_size,))

    # Estrae e sposta sul device
    x = X_data[ix].to(device)
    y = Y_data[ix].to(device)

    return x, y'''

def get_batch(split, train_data, val_data, batch_size, block_size, device):
    # Seleziona i dati corretti
    data = train_data if split == 'train' else val_data

    # Genera indici casuali
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    x,y = x.to(device), y.to(device)
    return x, y


@torch.no_grad()
def estimate_loss(model, eval_iters, train_data, val_data, batch_size, block_size, device):
    out = {}
    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            # Richiama get_batch passando i parametri ricevuti
            X, Y = get_batch(split, train_data, val_data, batch_size, block_size, device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()

    model.train()
    return out

In [4]:
iterations = 7  # Quante task diverse
for _ in range(iterations):
    current_string = replace_random_underscore(CURRENT_STRING, PROBABILITIES)

process = replace_underscores(current_string)
tree = PARSER.parse(process)

# ALBERO GIOCATTOLO
# Da rimuovere
tree = Tree('xor', [
    # 1. Primo figlio di XOR (T1)
    Tree('task', [Token('NAME', 'T1')]),

    # 2. Secondo figlio di XOR (Blocco Sequential)
    Tree('sequential', [

        # 2.1 Primo figlio di Sequential (Parallel)
        Tree('parallel', [
            # 2.1.1 Ramo sinistro del Parallel (Xor annidati)
            Tree('xor', [
                Tree('task', [Token('NAME', 'T2')]),
                Tree('xor', [
                    Tree('task', [Token('NAME', 'T3')]),
                    Tree('task', [Token('NAME', 'T4')])
                ])
            ]),
            # 2.1.2 Ramo destro del Parallel (Parallel T5, T6)
            Tree('parallel', [
                Tree('task', [Token('NAME', 'T5')]),
                Tree('task', [Token('NAME', 'T6')])
            ])
        ]),

        # 2.2 Secondo figlio di Sequential (Sequential T7, T8)
        Tree('sequential', [
            Tree('task', [Token('NAME', 'T7')]),
            Tree('task', [Token('NAME', 'T8')])
        ])
    ])
])

if NARY:
    tree = createNAryTree(tree)

# Oggetto PetriNetP - si inizializza automaticamente con il suo costruttore
net = PetriNetP(tree)

save_vis_petri_net(
    net.net,
    net.initial_marking,
    net.final_marking,
    FILE_PATH_PNG,
    format="png"  # Specifica il formato
)

# Oggetto Generator
generator = Generator(300, net)

# Creazione matrice identità delle regioni
df_region_identity = pd.DataFrame.from_dict(net.node_identity, orient='index').sort_index()
df_region_identity.columns = ['X', '+', '->']
print(df_region_identity)

# Creazione matrice regioni-figli per le regioni
df_region_children = pd.Series(net.node_children).explode()
df_region_children = pd.crosstab(df_region_children.index, df_region_children)
df_region_children = df_region_children.reindex(index=net.regions, columns=net.regions + net.tasks, fill_value=0)
df_region_children = df_region_children.astype(int)
df_region_children.index.name = None
df_region_children.columns.name = None
print(df_region_children)

traceEncoded_regions, traceEncoded_tasks = getEncoding(generator.generatedTraces, net.regions, net.tasks, net.open_clauses, net.end_clauses)

num_regions = len([i for i in traceEncoded_regions.index if str(i).startswith('R')])
num_tasks = len([i for i in traceEncoded_tasks.index if str(i).startswith('T')])

traceEncoded_regions.to_csv(TRACE_ENC_REG, index=True)
traceEncoded_tasks.to_csv(TRACE_ENC_TAS, index=True)

df_traces = pd.concat([traceEncoded_regions, traceEncoded_tasks], axis=0)
print(df_traces)

df_traces = df_traces.T

df_tracescopy = df_traces.copy()

unique_columns = df_traces.drop_duplicates()
unique_tuple = [tuple(x) for x in unique_columns.values]

# 2. Creiamo i dizionari di mappatura
# bit_to_id: trasforma la colonna di 12 bit in un numero
# id_to_bit: trasforma il numero nei 12 bit originali (per la generazione)
bit_to_id = {v: i for i, v in enumerate(unique_tuple)}
id_to_bit = {i: v for i, v in enumerate(unique_tuple)}

vocab_size = len(unique_columns)

encode = lambda a: [bit_to_id[tuple(x)] for x in a]
decode = lambda b: [id_to_bit[x] for x in b]

data = torch.tensor(encode(df_traces.values), dtype=torch.long)

#traces = cutTraces(traceEncoded_regions, traceEncoded_tasks)

block_size = 8
n_embd = 64
dropout = 0.4
n_head = 4
n_layer = 2
batch_size = 32
eval_iters = 200
eval_interval = 500
max_iters = 1500

#X, Y = create_training_set(traces,8)

model = BPMNTransformer(vocab_size, num_regions+num_tasks, block_size, n_embd, dropout, n_head, n_layer)
m = model.to(device)

print(sum(p.numel() for p in m.parameters()) / 1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

n = int(0.8 * len(df_traces))

train_data = data[:n]
val_data = data[n:]

#train_data_X = X[:n]
#train_data_Y = Y[:n]
#val_data_X = X[n:]
#val_data_Y = Y[n:]

'''xb, yb = get_batch('train', train_data, val_data, batch_size, block_size, device)
print(xb)
print(yb)'''

    X  +  ->
R0  1  0   0
R1  0  0   1
R2  0  1   0
R3  1  0   0
    R0  R1  R2  R3  T1  T2  T3  T4  T5  T6  T7  T8
R0   0   1   0   0   1   0   0   0   0   0   0   0
R1   0   0   1   0   0   0   0   0   0   0   1   1
R2   0   0   0   1   0   0   0   0   1   1   0   0
R3   0   0   0   0   0   1   1   1   0   0   0   0
    0     1     2     3     4     5     6     7     8     9     ...  1854  \
R0     1     0     1     0     1     1     1     1     1     1  ...     1   
R1     0     0     0     0     1     1     1     1     1     1  ...     1   
R2     0     0     0     0     1     1     1     1     1     0  ...     1   
R3     0     0     0     0     0     0     0     1     1     0  ...     0   
T1     1     0     1     0     0     0     0     0     0     0  ...     0   
T2     0     0     0     0     0     0     0     1     1     0  ...     0   
T3     0     0     0     0     0     0     0     0     0     0  ...     0   
T4     0     0     0     0     0     0     0     0     0     0  

"xb, yb = get_batch('train', train_data, val_data, batch_size, block_size, device)\nprint(xb)\nprint(yb)"

In [5]:
# Dentro il ciclo for iter in range(max_iters):
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model, eval_iters, train_data, val_data, batch_size, block_size, device)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # A. Pesca i dati (usando la funzione esterna)
    xb, yb = get_batch('train', train_data, val_data, batch_size, block_size, device)

    # B. FORWARD PASS: Il modello "gira" e produce una previsione
    logits, loss = model(xb, yb)

    # C. RESET GRADIENTI: Puliamo i calcoli del giro precedente
    optimizer.zero_grad(set_to_none=True)

    # D. BACKWARD PASS: Calcoliamo l'errore per ogni neurone (L'Anima del training)
    loss.backward()

    # E. OPTIMIZER STEP: Aggiorniamo i pesi per sbagliare meno al prossimo giro
    optimizer.step()

step 0: train loss 3.0855, val loss 3.0787
step 500: train loss 0.7441, val loss 0.7917
step 1000: train loss 0.6567, val loss 0.7454
step 1499: train loss 0.6265, val loss 0.7325


In [8]:
context = torch.tensor(encode([[0]*(num_regions+num_tasks)]), dtype=torch.long, device=device).unsqueeze(0)
print(context)
# Generiamo gli ID
generated_indices = m.generate(idx=context, block_size=block_size, max_new_tokens=20)[0].tolist()

# Decodifichiamo in bit
decoded_output = decode(generated_indices)

print("\n--- TRACCIA GENERATA ---")
for i, step in enumerate(decoded_output):
    # Trasformiamo i valori np.int64 in semplici interi e poi in stringa per leggerli meglio
    bit_list = [int(b) for b in step]
    print(f"Step {i:02d}: {bit_list}")

tensor([[1]], device='cuda:0')

--- TRACCIA GENERATA ---
Step 00: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Step 01: [1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
Step 02: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Step 03: [1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0]
Step 04: [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Step 05: [1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0]
Step 06: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Step 07: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]
Step 08: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Step 09: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
Step 10: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Step 11: [1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0]
Step 12: [1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0]
Step 13: [1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0]
Step 14: [1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0]
Step 15: [1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0]
Step 16: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Step 17: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]
Step 18: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Step 19: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
Step 20: [0, 0, 0, 0, 0